# CVPR_CH4CHL - Data Preprocessing & Feature Packing

**Platform**: Colab (Google Drive mount)
**Runtime**: CPU
**Estimated time**: ~10-15 min

## Goals
1. Extract 23 gait keypoints from 339K JSON frames → per-video numpy arrays
2. Compute joint angles (knee, ankle, hip, trunk) per frame
3. Detect gait cycles (heel strikes) per sagittal video
4. Compute handcrafted gait features per patient
5. Pack into three `.pkl` files written to Drive:
   - `keypoints.pkl` — raw keypoint sequences per video
   - `features.pkl` — joint angles + gait parameters per patient
   - `labels.pkl` — Track 1 + Track 2 labels

Downstream training notebooks load these via `pd.read_pickle()`; no further JSON parsing required.

In [ ]:
# === Cell 1: Setup & Imports ===
import json
import os
import pickle
import time
import numpy as np
import pandas as pd
from collections import defaultdict
from pathlib import Path
from scipy.signal import find_peaks, butter, filtfilt
import warnings
warnings.filterwarnings('ignore')

print('Imports OK')

In [ ]:
# === Cell 2: Mount Drive & Config ===
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    DRIVE_ROOT = '/content/drive/MyDrive/CVPR_CH4CHL'
    DRIVE_DATA = f'{DRIVE_ROOT}/1_data/raw'
    LOCAL_DATA = '/content/data'
    os.makedirs(LOCAL_DATA, exist_ok=True)

    # Copy labels
    for f in ['track1_train.json', 'track2_train.json']:
        src, dst = f'{DRIVE_DATA}/{f}', f'{LOCAL_DATA}/{f}'
        if not os.path.exists(dst):
            os.system(f'cp "{src}" "{dst}"')

    # Extract dataset.tar.bz2 to local
    DATASET_DIR = f'{LOCAL_DATA}/dataset'
    TAR_FILE = f'{DRIVE_DATA}/dataset.tar.bz2'
    if not os.path.exists(DATASET_DIR):
        assert os.path.exists(TAR_FILE), f'Missing: {TAR_FILE}'
        print('Extracting dataset.tar.bz2 to local disk...')
        t0 = time.time()
        os.system(f'tar -xjf "{TAR_FILE}" -C "{LOCAL_DATA}"')
        # Handle nested extraction
        if not os.path.exists(DATASET_DIR):
            for d in os.listdir(LOCAL_DATA):
                p = f'{LOCAL_DATA}/{d}'
                if os.path.isdir(p) and d not in ('dataset',):
                    os.rename(p, DATASET_DIR)
                    break
        print(f'Done in {time.time()-t0:.0f}s')

    TRACK1_JSON = f'{LOCAL_DATA}/track1_train.json'
    TRACK2_JSON = f'{LOCAL_DATA}/track2_train.json'

    # Output goes to Drive (persistent)
    PROCESSED_DIR = f'{DRIVE_ROOT}/1_data/processed'
else:
    DRIVE_ROOT = os.environ.get(
        'CV4CHL_REPO_ROOT',
        os.getcwd(),
    )
    DATA_DIR = f'{DRIVE_ROOT}/1_data/raw'
    DATASET_DIR = f'{DATA_DIR}/dataset'
    TRACK1_JSON = f'{DATA_DIR}/track1_train.json'
    TRACK2_JSON = f'{DATA_DIR}/track2_train.json'
    PROCESSED_DIR = f'{DRIVE_ROOT}/1_data/processed'

os.makedirs(PROCESSED_DIR, exist_ok=True)

for p in [TRACK1_JSON, TRACK2_JSON, DATASET_DIR]:
    assert os.path.exists(p), f'Missing: {p}'

print(f'Platform: {"Colab" if IN_COLAB else "Local"}')
print(f'Dataset: {DATASET_DIR}')
print(f'Output: {PROCESSED_DIR}')

In [ ]:
# === Cell 3: Constants & Keypoint Definitions ===

# COCO-WholeBody gait-relevant indices (23 keypoints)
BODY_IDX = list(range(0, 17))
FEET_IDX = list(range(17, 23))
GAIT_IDX = BODY_IDX + FEET_IDX  # 23 keypoints

GAIT_KP_NAMES = [
    'nose', 'left_eye', 'right_eye', 'left_ear', 'right_ear',
    'left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow',
    'left_wrist', 'right_wrist', 'left_hip', 'right_hip',
    'left_knee', 'right_knee', 'left_ankle', 'right_ankle',
    'left_big_toe', 'left_small_toe', 'left_heel',
    'right_big_toe', 'right_small_toe', 'right_heel',
]
KP = {name: i for i, name in enumerate(GAIT_KP_NAMES)}

# EVGS item -> view plane mapping
EVGS_SAGITTAL_ITEMS = [1, 2, 3, 6, 7, 8, 9, 10, 11, 12, 16, 17]  # 12 items
EVGS_CORONAL_ITEMS = [4, 5, 13, 14, 15]  # 5 items

# Test IDs (canonical CVPR 2026 challenge cohort).
TRACK1_TEST_IDS = [4, 5, 18, 26, 28, 40, 42, 43, 47, 48, 53, 54, 72, 78, 83, 85]
TRACK2_TEST_IDS = [4, 6, 7, 13, 26, 35, 39, 42, 50]

# Verify the canonical lists against the Kaggle-shipped sample submission
# when it is available locally. The hardcoded lists above are the source of
# truth for offline reproduction; this assertion catches the case where the
# lists drift out of sync with a later sample submission file.
_sample_sub_path = None
for _candidate in ('1_data/raw/sample_submission.csv',
                   '1_data/raw/sample_submission_track1.csv'):
    _path = os.path.join(DRIVE_ROOT if 'DRIVE_ROOT' in dir() else os.getcwd(), _candidate)
    if os.path.exists(_path):
        _sample_sub_path = _path
        break
if _sample_sub_path is not None:
    _sample = pd.read_csv(_sample_sub_path)
    _t1_inferred = sorted({int(s.split('-')[1]) for s in _sample['ID'].astype(str)
                           if s.startswith('track1-')})
    _t2_inferred = sorted({int(s.split('-')[1]) for s in _sample['ID'].astype(str)
                           if s.startswith('track2-')})
    if _t1_inferred:
        assert sorted(TRACK1_TEST_IDS) == _t1_inferred, (
            f'TRACK1_TEST_IDS does not match {_sample_sub_path}: '
            f'{sorted(TRACK1_TEST_IDS)} vs {_t1_inferred}')
    if _t2_inferred:
        assert sorted(TRACK2_TEST_IDS) == _t2_inferred, (
            f'TRACK2_TEST_IDS does not match {_sample_sub_path}: '
            f'{sorted(TRACK2_TEST_IDS)} vs {_t2_inferred}')
    print(f'Test IDs verified against {_sample_sub_path}')
else:
    print('sample_submission.csv not found; using hardcoded canonical test IDs')

print(f'Gait keypoints: {len(GAIT_IDX)}')
print(f'EVGS sagittal items: {EVGS_SAGITTAL_ITEMS}')
print(f'EVGS coronal items: {EVGS_CORONAL_ITEMS}')

In [ ]:
# === Cell 4: Core Functions — Keypoint Loading ===

def load_video_keypoints(video_path):
    """Load all frames, extract gait keypoints.

    Returns:
        keypoints: (n_frames, 23, 2) float32
        scores: (n_frames, 23) float32
        n_frames: int
    """
    frame_files = sorted([f for f in os.listdir(video_path) if f.endswith('.json')])
    if not frame_files:
        return np.empty((0, 23, 2), dtype=np.float32), np.empty((0, 23), dtype=np.float32), 0

    kps_list, sc_list = [], []
    for fname in frame_files:
        with open(os.path.join(video_path, fname)) as f:
            data = json.load(f)
        if not data['instance_info']:
            continue
        kps = np.array(data['instance_info'][0]['keypoints'], dtype=np.float32)
        scs = np.array(data['instance_info'][0]['keypoint_scores'], dtype=np.float32)
        kps_list.append(kps[GAIT_IDX])
        sc_list.append(scs[GAIT_IDX])

    if not kps_list:
        return np.empty((0, 23, 2), dtype=np.float32), np.empty((0, 23), dtype=np.float32), 0

    return np.array(kps_list), np.array(sc_list), len(kps_list)


def parse_video_name(video_name):
    """Parse '{pid}-{session}_{view}_{start-end}_filtered_pose' format."""
    parts = video_name.split('_')
    pid_session = parts[0]  # e.g. '0001-0001'
    view = parts[1] if len(parts) >= 2 else 'unknown'
    return {
        'pid_session': pid_session,
        'session': pid_session.split('-')[1] if '-' in pid_session else '0001',
        'view': view,
    }

print('Loading functions OK')

In [ ]:
# === Cell 5: Core Functions — Joint Angles & Gait Cycles ===

def compute_angle(p1, p2, p3):
    """Angle at p2 formed by p1-p2-p3, in degrees."""
    v1 = p1 - p2
    v2 = p3 - p2
    cos_a = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-8)
    return np.degrees(np.arccos(np.clip(cos_a, -1, 1)))


def compute_joint_angles_sequence(kps, side='left'):
    """Compute joint angles for all frames.

    Args:
        kps: (T, 23, 2)
        side: 'left' or 'right'

    Returns:
        dict with keys 'knee', 'ankle', 'hip', 'trunk', each (T,) array
    """
    T = len(kps)
    angles = {k: np.zeros(T, dtype=np.float32) for k in ['knee', 'ankle', 'hip', 'trunk']}

    for t in range(T):
        k = kps[t]
        s = side

        # Knee: hip-knee-ankle
        angles['knee'][t] = compute_angle(k[KP[f'{s}_hip']], k[KP[f'{s}_knee']], k[KP[f'{s}_ankle']])
        # Ankle: knee-ankle-big_toe
        angles['ankle'][t] = compute_angle(k[KP[f'{s}_knee']], k[KP[f'{s}_ankle']], k[KP[f'{s}_big_toe']])
        # Hip: shoulder-hip-knee
        angles['hip'][t] = compute_angle(k[KP[f'{s}_shoulder']], k[KP[f'{s}_hip']], k[KP[f'{s}_knee']])
        # Trunk: midshoulder-midhip vs vertical
        ms = (k[KP['left_shoulder']] + k[KP['right_shoulder']]) / 2
        mh = (k[KP['left_hip']] + k[KP['right_hip']]) / 2
        trunk_vec = ms - mh
        vertical = np.array([0, -1], dtype=np.float32)
        cos_a = np.dot(trunk_vec, vertical) / (np.linalg.norm(trunk_vec) + 1e-8)
        angles['trunk'][t] = np.degrees(np.arccos(np.clip(cos_a, -1, 1)))

    return angles


def smooth_signal(signal, kernel_size=5):
    """Simple moving average smoothing."""
    if len(signal) <= kernel_size:
        return signal.copy()
    kernel = np.ones(kernel_size) / kernel_size
    return np.convolve(signal, kernel, mode='same')


def detect_heel_strikes(kps, side='left', min_distance=15, min_prominence=5):
    """Detect heel strikes in sagittal view using heel-midhip x-distance.

    Returns:
        hs_indices: array of heel strike frame indices
        signal_smooth: the smoothed distance signal
    """
    heel_idx = KP[f'{side}_heel']
    T = len(kps)
    signal = np.zeros(T, dtype=np.float32)

    for t in range(T):
        midhip = (kps[t, KP['left_hip']] + kps[t, KP['right_hip']]) / 2
        signal[t] = kps[t, heel_idx, 0] - midhip[0]

    signal_smooth = smooth_signal(signal)
    hs_indices, _ = find_peaks(signal_smooth, distance=min_distance, prominence=min_prominence)

    return hs_indices, signal_smooth


def segment_gait_cycles(kps, side='left'):
    """Segment a video into gait cycles (heel-strike to heel-strike).

    Returns:
        list of (start_frame, end_frame) tuples
    """
    hs, _ = detect_heel_strikes(kps, side=side)
    cycles = []
    for i in range(len(hs) - 1):
        cycles.append((int(hs[i]), int(hs[i + 1])))
    return cycles

print('Angle & gait cycle functions OK')

In [ ]:
# === Cell 6: Core Functions — Handcrafted Gait Features ===

def extract_gait_features_from_angles(angles_dict, gait_cycles=None):
    """Extract statistical gait features from joint angle sequences.

    For each angle (knee, ankle, hip, trunk), compute:
    - mean, std, min, max, range
    - If gait cycles provided: per-cycle stats averaged across cycles

    Args:
        angles_dict: dict with keys 'knee', 'ankle', 'hip', 'trunk', each (T,) array
        gait_cycles: optional list of (start, end) tuples

    Returns:
        dict of feature_name -> float
    """
    features = {}

    for angle_name, signal in angles_dict.items():
        if len(signal) == 0:
            for stat in ['mean', 'std', 'min', 'max', 'range']:
                features[f'{angle_name}_{stat}'] = 0.0
            continue

        features[f'{angle_name}_mean'] = float(np.mean(signal))
        features[f'{angle_name}_std'] = float(np.std(signal))
        features[f'{angle_name}_min'] = float(np.min(signal))
        features[f'{angle_name}_max'] = float(np.max(signal))
        features[f'{angle_name}_range'] = float(np.ptp(signal))

        # Per-cycle features (more robust to variable-length videos)
        if gait_cycles and len(gait_cycles) > 0:
            cycle_ranges, cycle_means = [], []
            for start, end in gait_cycles:
                if end <= len(signal):
                    seg = signal[start:end]
                    cycle_ranges.append(float(np.ptp(seg)))
                    cycle_means.append(float(np.mean(seg)))
            if cycle_ranges:
                features[f'{angle_name}_cycle_range_mean'] = float(np.mean(cycle_ranges))
                features[f'{angle_name}_cycle_range_std'] = float(np.std(cycle_ranges))
                features[f'{angle_name}_cycle_mean_std'] = float(np.std(cycle_means))
            else:
                features[f'{angle_name}_cycle_range_mean'] = features[f'{angle_name}_range']
                features[f'{angle_name}_cycle_range_std'] = 0.0
                features[f'{angle_name}_cycle_mean_std'] = 0.0
        else:
            features[f'{angle_name}_cycle_range_mean'] = features[f'{angle_name}_range']
            features[f'{angle_name}_cycle_range_std'] = 0.0
            features[f'{angle_name}_cycle_mean_std'] = 0.0

    return features


def extract_spatial_features(kps):
    """Extract spatial features from keypoint sequence.

    Args:
        kps: (T, 23, 2)

    Returns:
        dict of feature_name -> float
    """
    features = {}
    T = len(kps)
    if T == 0:
        return features

    # Stride length proxy: max distance between left/right heels in x
    lh = kps[:, KP['left_heel'], 0]
    rh = kps[:, KP['right_heel'], 0]
    heel_dist = np.abs(lh - rh)
    features['heel_x_dist_mean'] = float(np.mean(heel_dist))
    features['heel_x_dist_max'] = float(np.max(heel_dist))

    # Body height proxy: shoulder-ankle distance (y)
    for side in ['left', 'right']:
        sh_y = kps[:, KP[f'{side}_shoulder'], 1]
        an_y = kps[:, KP[f'{side}_ankle'], 1]
        height = np.abs(an_y - sh_y)
        features[f'{side}_body_height_mean'] = float(np.mean(height))

    # Hip width: left_hip to right_hip distance
    hip_dist = np.linalg.norm(kps[:, KP['left_hip']] - kps[:, KP['right_hip']], axis=1)
    features['hip_width_mean'] = float(np.mean(hip_dist))

    # Walking speed proxy: midhip x displacement per frame
    midhip = (kps[:, KP['left_hip']] + kps[:, KP['right_hip']]) / 2
    if T > 1:
        dx = np.abs(np.diff(midhip[:, 0]))
        features['walk_speed_mean'] = float(np.mean(dx))
        features['walk_speed_std'] = float(np.std(dx))
    else:
        features['walk_speed_mean'] = 0.0
        features['walk_speed_std'] = 0.0

    # Vertical oscillation of midhip (gait bounce)
    if T > 1:
        features['midhip_y_std'] = float(np.std(midhip[:, 1]))
    else:
        features['midhip_y_std'] = 0.0

    return features

print('Feature extraction functions OK')

## Step 1: Extract all keypoints → `keypoints.pkl`

Scan every video for all 110 patients, extract 23 gait keypoints, and store as:
```python
{
    (patient_id, video_name): {
        'keypoints': np.array (T, 23, 2),
        'scores': np.array (T, 23),
        'view': str,
        'session': str,
        'n_frames': int,
    }
}
```

In [ ]:
# === Cell 7: Extract Keypoints from All Videos ===
t0 = time.time()
keypoints_db = {}
total_frames = 0
total_videos = 0

patient_dirs = sorted([d for d in os.listdir(DATASET_DIR) if d.isdigit()])
print(f'Processing {len(patient_dirs)} patients...')

for i, pid_str in enumerate(patient_dirs):
    pid = int(pid_str)
    pid_path = os.path.join(DATASET_DIR, pid_str)
    videos = sorted([v for v in os.listdir(pid_path)
                     if os.path.isdir(os.path.join(pid_path, v))])

    for video_name in videos:
        video_path = os.path.join(pid_path, video_name)
        kps, scores, n_frames = load_video_keypoints(video_path)

        if n_frames == 0:
            continue

        meta = parse_video_name(video_name)
        keypoints_db[(pid, video_name)] = {
            'keypoints': kps,
            'scores': scores,
            'view': meta['view'],
            'session': meta['session'],
            'n_frames': n_frames,
        }
        total_frames += n_frames
        total_videos += 1

    if (i + 1) % 20 == 0:
        print(f'  {i+1}/{len(patient_dirs)} patients, {total_videos} videos, {total_frames:,} frames')

elapsed = time.time() - t0
print(f'\nDone: {total_videos} videos, {total_frames:,} frames in {elapsed:.0f}s')
print(f'DB size in memory: ~{sum(v["keypoints"].nbytes + v["scores"].nbytes for v in keypoints_db.values()) / 1e6:.0f} MB')

In [ ]:
# === Cell 8: Save keypoints.pkl to Drive ===
kp_path = os.path.join(PROCESSED_DIR, 'keypoints.pkl')
print(f'Saving keypoints to {kp_path}...')
with open(kp_path, 'wb') as f:
    pickle.dump(keypoints_db, f, protocol=pickle.HIGHEST_PROTOCOL)
size_mb = os.path.getsize(kp_path) / 1e6
print(f'Saved: {size_mb:.1f} MB ({len(keypoints_db)} videos)')

## Step 2: Compute features per patient → `features.pkl`

Per patient, per side (left/right):
- Aggregate joint angles from all sagittal videos
- Aggregate spatial features from all coronal videos
- Detect gait cycles from sagittal videos
- Output: one feature vector per (patient_id, side)

In [ ]:
# === Cell 9: Compute Features Per Patient ===
# Group videos by patient
from collections import defaultdict

patient_videos = defaultdict(list)
for (pid, vname), data in keypoints_db.items():
    patient_videos[pid].append((vname, data))

feature_rows = []
all_patient_ids = sorted(patient_videos.keys())
print(f'Computing features for {len(all_patient_ids)} patients...')

for pid in all_patient_ids:
    videos = patient_videos[pid]

    for side in ['left', 'right']:
        feat = {'patient_id': pid, 'side': side}

        # --- Sagittal views: joint angles + gait cycles ---
        sagittal_angles = {k: [] for k in ['knee', 'ankle', 'hip', 'trunk']}
        all_gait_cycles_angles = {k: [] for k in ['knee', 'ankle', 'hip', 'trunk']}
        total_cycles = 0

        for vname, data in videos:
            if data['view'] not in ('left', 'right'):
                continue
            kps = data['keypoints']
            if len(kps) < 20:
                continue

            angles = compute_joint_angles_sequence(kps, side=side)
            cycles = segment_gait_cycles(kps, side=side)
            total_cycles += len(cycles)

            for k in sagittal_angles:
                sagittal_angles[k].append(angles[k])

            # Per-cycle angle stats
            for start, end in cycles:
                for k in sagittal_angles:
                    if end <= len(angles[k]):
                        all_gait_cycles_angles[k].append(angles[k][start:end])

        # Concatenate all sagittal video angles
        concat_angles = {}
        for k in sagittal_angles:
            if sagittal_angles[k]:
                concat_angles[k] = np.concatenate(sagittal_angles[k])
            else:
                concat_angles[k] = np.array([], dtype=np.float32)

        # Extract angle features
        angle_feats = extract_gait_features_from_angles(
            concat_angles,
            gait_cycles=[(0, len(c)) for c in all_gait_cycles_angles.get('knee', [])]
            if all_gait_cycles_angles.get('knee') else None
        )
        # Prefix with side
        for k, v in angle_feats.items():
            feat[f'sag_{k}'] = v

        feat['n_gait_cycles'] = total_cycles

        # --- Per-cycle normalized angle features ---
        # Resample each cycle to 100 points, compute mean cycle shape
        for angle_name in ['knee', 'ankle', 'hip']:
            cycle_list = all_gait_cycles_angles.get(angle_name, [])
            if cycle_list and len(cycle_list) >= 1:
                resampled = []
                for cyc in cycle_list:
                    if len(cyc) >= 5:
                        x = np.linspace(0, 1, len(cyc))
                        x_new = np.linspace(0, 1, 50)
                        resampled.append(np.interp(x_new, x, cyc))
                if resampled:
                    mean_cycle = np.mean(resampled, axis=0)
                    feat[f'sag_{angle_name}_cycle_max'] = float(np.max(mean_cycle))
                    feat[f'sag_{angle_name}_cycle_min'] = float(np.min(mean_cycle))
                    feat[f'sag_{angle_name}_cycle_range'] = float(np.ptp(mean_cycle))
                    # Phase of max (% of gait cycle)
                    feat[f'sag_{angle_name}_cycle_max_phase'] = float(np.argmax(mean_cycle) / 50)
                    feat[f'sag_{angle_name}_cycle_min_phase'] = float(np.argmin(mean_cycle) / 50)
                    continue
            # Fallback
            for suffix in ['_cycle_max', '_cycle_min', '_cycle_range', '_cycle_max_phase', '_cycle_min_phase']:
                feat[f'sag_{angle_name}{suffix}'] = 0.0

        # --- Coronal views: spatial features ---
        coronal_spatial = []
        for vname, data in videos:
            if data['view'] not in ('forward', 'backward'):
                continue
            kps = data['keypoints']
            if len(kps) < 10:
                continue
            coronal_spatial.append(extract_spatial_features(kps))

        if coronal_spatial:
            # Average spatial features across coronal videos
            all_keys = coronal_spatial[0].keys()
            for k in all_keys:
                vals = [d[k] for d in coronal_spatial if k in d]
                feat[f'cor_{k}'] = float(np.mean(vals)) if vals else 0.0
        else:
            # Fallback: use sagittal views for spatial features
            for vname, data in videos:
                if data['view'] in ('left', 'right') and len(data['keypoints']) >= 10:
                    sp = extract_spatial_features(data['keypoints'])
                    for k, v in sp.items():
                        feat[f'cor_{k}'] = v
                    break

        # --- All views: aggregate spatial features ---
        all_spatial = []
        for vname, data in videos:
            if len(data['keypoints']) >= 10:
                all_spatial.append(extract_spatial_features(data['keypoints']))
        if all_spatial:
            for k in all_spatial[0].keys():
                vals = [d[k] for d in all_spatial if k in d]
                feat[f'all_{k}'] = float(np.mean(vals)) if vals else 0.0

        # Video count features
        feat['n_sagittal_videos'] = sum(1 for _, d in videos if d['view'] in ('left', 'right'))
        feat['n_coronal_videos'] = sum(1 for _, d in videos if d['view'] in ('forward', 'backward'))
        feat['n_total_videos'] = len(videos)
        feat['n_total_frames'] = sum(d['n_frames'] for _, d in videos)

        feature_rows.append(feat)

df_features = pd.DataFrame(feature_rows)
print(f'\nFeature matrix: {df_features.shape}')
print(f'Columns: {len(df_features.columns)}')
print(f'Sample:\n{df_features.head(2).T}')

In [ ]:
# === Cell 10: Save features.pkl ===
feat_path = os.path.join(PROCESSED_DIR, 'features.pkl')
df_features.to_pickle(feat_path)
print(f'Saved features: {feat_path} ({os.path.getsize(feat_path)/1e3:.0f} KB)')
print(f'Shape: {df_features.shape} — {df_features.shape[0]} rows (patient×side), {df_features.shape[1]} columns')

## Step 3: Pack labels → `labels.pkl`

Merge Track 1 + Track 2 labels into a unified format.

In [ ]:
# === Cell 11: Pack Labels ===
with open(TRACK1_JSON) as f:
    track1_raw = json.load(f)
with open(TRACK2_JSON) as f:
    track2_raw = json.load(f)

# Track 1: flatten to (patient_id, side) rows
t1_rows = []
for p in track1_raw:
    pid = p['patient_id']
    for side in ['left', 'right']:
        row = {'patient_id': pid, 'side': side, 'track': 1}
        for item in range(1, 18):
            row[f'evgs_{item}'] = p[side][str(item)]
        row['evgs_total'] = p[side]['Total']
        t1_rows.append(row)

df_t1_labels = pd.DataFrame(t1_rows)

# Track 2: flatten to (patient_id, side) rows
t2_rows = []
for p in track2_raw:
    pid = p['patient_id']
    for side in ['left', 'right']:
        row = {
            'patient_id': pid,
            'side': side,
            'track': 2,
            'gait_subtype': p[side]['gait_subtype'],
        }
        t2_rows.append(row)

df_t2_labels = pd.DataFrame(t2_rows)

labels = {
    'track1': df_t1_labels,
    'track2': df_t2_labels,
    'track1_test_ids': TRACK1_TEST_IDS,
    'track2_test_ids': TRACK2_TEST_IDS,
}

label_path = os.path.join(PROCESSED_DIR, 'labels.pkl')
with open(label_path, 'wb') as f:
    pickle.dump(labels, f)

print(f'Track 1 labels: {df_t1_labels.shape} — {df_t1_labels["patient_id"].nunique()} patients')
print(f'Track 2 labels: {df_t2_labels.shape} — {df_t2_labels["patient_id"].nunique()} patients')
print(f'Saved: {label_path}')

## Step 4: Merge features + labels → `train_ready.pkl`

Combine features and labels into a single DataFrame ready for training.

In [ ]:
# === Cell 12: Merge Features + Labels → train_ready.pkl ===

# Track 1: merge features + labels
df_t1 = df_features.merge(df_t1_labels, on=['patient_id', 'side'], how='inner')
print(f'Track 1 train-ready: {df_t1.shape} ({df_t1["patient_id"].nunique()} patients)')

# Track 2: merge features + labels
df_t2 = df_features.merge(df_t2_labels, on=['patient_id', 'side'], how='inner')
print(f'Track 2 train-ready: {df_t2.shape} ({df_t2["patient_id"].nunique()} patients)')

# Also prepare test features (patients with features but no labels)
t1_train_pids = set(df_t1_labels['patient_id'].unique())
t2_train_pids = set(df_t2_labels['patient_id'].unique())

df_t1_test_feats = df_features[df_features['patient_id'].isin(TRACK1_TEST_IDS)]
df_t2_test_feats = df_features[df_features['patient_id'].isin(TRACK2_TEST_IDS)]
print(f'\nTrack 1 test features: {df_t1_test_feats.shape} (IDs: {sorted(df_t1_test_feats["patient_id"].unique())})')
print(f'Track 2 test features: {df_t2_test_feats.shape} (IDs: {sorted(df_t2_test_feats["patient_id"].unique())})')

# Save
train_ready = {
    'track1_train': df_t1,
    'track2_train': df_t2,
    'track1_test': df_t1_test_feats,
    'track2_test': df_t2_test_feats,
    'all_features': df_features,
}
ready_path = os.path.join(PROCESSED_DIR, 'train_ready.pkl')
with open(ready_path, 'wb') as f:
    pickle.dump(train_ready, f)
print(f'\nSaved: {ready_path} ({os.path.getsize(ready_path)/1e3:.0f} KB)')

In [ ]:
# === Cell 13: Verification — Check Output Files ===
print('=== Output Files ===')
for fname in ['keypoints.pkl', 'features.pkl', 'labels.pkl', 'train_ready.pkl']:
    fpath = os.path.join(PROCESSED_DIR, fname)
    if os.path.exists(fpath):
        size = os.path.getsize(fpath)
        print(f'  {fname}: {size/1e6:.1f} MB' if size > 1e6 else f'  {fname}: {size/1e3:.0f} KB')
    else:
        print(f'  {fname}: MISSING!')

print(f'\n=== Quick Sanity Check ===')
# Reload and verify
with open(os.path.join(PROCESSED_DIR, 'train_ready.pkl'), 'rb') as f:
    check = pickle.load(f)

df_check = check['track1_train']
# Verify feature columns exist
feat_cols = [c for c in df_check.columns if c.startswith('sag_') or c.startswith('cor_') or c.startswith('all_')]
label_cols = [c for c in df_check.columns if c.startswith('evgs_')]
print(f'Track 1: {len(feat_cols)} feature columns, {len(label_cols)} label columns')
print(f'Feature columns sample: {feat_cols[:5]}')
print(f'Label columns: {label_cols}')
print(f'NaN count: {df_check[feat_cols].isna().sum().sum()}')

df_check2 = check['track2_train']
print(f'\nTrack 2: {df_check2.shape[0]} samples, subtypes: {df_check2["gait_subtype"].value_counts().to_dict()}')

print('\n=== All Done! ===')